In [1]:
import mlflow
import optuna
import mlflow.sklearn
from imblearn.over_sampling import SMOTE, ADASYN
from imblearn.under_sampling import RandomUnderSampler
from imblearn.combine import SMOTEENN
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.model_selection import train_test_split, cross_val_predict, StratifiedKFold
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

In [2]:
import os
from dotenv import load_dotenv

load_dotenv()

dagshub_token = os.getenv("DAGSHUB_PAT")

if not dagshub_token:
    raise EnvironmentError("DAGSHUB_PAT environment variable is not set")

# DagsHub credentials for MLflow
os.environ["MLFLOW_TRACKING_USERNAME"] = "rajeshxdatascience"
os.environ["MLFLOW_TRACKING_PASSWORD"] = dagshub_token

# MLflow tracking URI
repo_owner = "rajeshxdatascience"
repo_name = "yt-comment-sentiment-analysis"

mlflow.set_tracking_uri(
    f"https://dagshub.com/{repo_owner}/{repo_name}.mlflow"
)

In [3]:
df = pd.read_csv(r"C:\Users\rajes\Desktop\yt-comment-sentiment-analysis\data\processed\reddit_preprocessing.csv").dropna(subset=['clean_comment'])
df.head()

,clean_comment,category
0,family mormon never tried explain still stare ...,1
1,buddhism much lot compatible christianity espe...,1
2,seriously say thing first get complex explain ...,-1
3,learned want teach different focus goal not wr...,0
4,benefit may want read living buddha living chr...,1


In [4]:
df.shape

(36662, 2)

In [5]:
mlflow.set_experiment("Exp 6 - LightGBM HP Tuning")

<Experiment: artifact_location='mlflow-artifacts:/83e51e4202b24e5fae4b828af513793f', creation_time=1786644864463, effective_trace_archival_retention=None, experiment_id='6', last_update_time=1786644864463, lifecycle_stage='active', name='Exp 6 - LightGBM HP Tuning', tags={'mlflow.experimentKind': 'custom_model_development'}, trace_location=None, workspace='default'>

In [6]:
# Step 1: Remap the class labels from [-1, 0, 1] to [2, 0, 1]
df['category'] = df['category'].map({-1: 2, 0: 0, 1: 1})

# Step 2: Remove rows where the target labels (category) are NaN

df = df.dropna(subset=['category'])

# Step 3: Train-Test Split FIRST
X_train_text, X_test_text, y_train, y_test = train_test_split(df['clean_comment'],df['category'],test_size=0.2,random_state=42,stratify=df['category'])

# Step 4: TF-IDF Vectorizer
ngram_range = (1, 3)  # Trigram
max_features = 10000

vectorizer = TfidfVectorizer(ngram_range=ngram_range,max_features=max_features)

# Fit ONLY on training data
X_train = vectorizer.fit_transform(X_train_text)

# Only transform test data
X_test = vectorizer.transform(X_test_text)

# Step 5: Apply ADASYN ONLY on training data
adasyn = ADASYN(random_state=42)
X_train_resampled, y_train_resampled = adasyn.fit_resample(X_train,y_train)

# Function to log results in MLFLOW
def log_mlflow(model_name, model, X_train_resampled, X_test, y_train_resampled, y_test, params, trial_number):
    with mlflow.start_run():
        # Log model type
        mlflow.set_tag("mlflow.runName", f"Trial_{trial_number}_{model_name}_SMOTE_TFIDF_Trigrams")
        mlflow.set_tag("experiment_type", "algoritm_comparison")

        # Log algorithm name as a paramter
        mlflow.log_param("algo_name", model_name)

        # Log hyperparameters
        for key, value in params.items():
            mlflow.log_param(key, value)

        # Train model
        model.fit(X_train_resampled, y_train_resampled)
        y_pred = model.predict(X_test)
                
        # Log accuracy
        accuracy = accuracy_score(y_test, y_pred)
        mlflow.log_metric("accuracy", accuracy)
                
        # Log Classification report
        classification_rep = classification_report(y_test, y_pred, output_dict=True)
                        
        for label, metrics in classification_rep.items():
            if isinstance(metrics, dict):
                for metric, value in metrics.items():
                     mlflow.log_metric(f"{label}_{metric}", value)
                
        # Log the model
        mlflow.lightgbm.log_model(model,f"{model_name}_model")

    return accuracy

# Step 6: Optuna objective function for XGBoost
def objective_lightgbm(trial):
    n_estimators = trial.suggest_int('n_estimators', 50, 300)
    learning_rate = trial.suggest_float('learning_rate', 1e-4, 1e-1, log=True)
    max_depth = trial.suggest_int('max_depth', 3, 10)
    num_leaves = trial.suggest_int('num_leaves', 20, 150)
    min_child_samples = trial.suggest_int('min_child_samples', 10, 100)
    colsample_bytree = trial.suggest_float('colsample_bytree', 0.5, 1.0)
    subsample = trial.suggest_float('subsample', 0.5, 1.0)
    reg_alpha = trial.suggest_float('reg_alpha', 1e-4, 10.0, log=True) # L1 regularization
    reg_lambda = trial.suggest_float('reg_lambda', 1e-4, 10.0, log=True) # L2 regularization

    params = {
        "n_estimators": n_estimators,
        "learning_rate": learning_rate,
        "max_depth": max_depth,
        "num_leaves": num_leaves,
        "min_child_samples": min_child_samples,
        "colsample_bytree": colsample_bytree,
        "subsample": subsample,
        "subsample_freq": 1,
        "reg_alpha": reg_alpha,
        "reg_lambda": reg_lambda,
        "random_state": 42,
        "verbosity": -1
    }

    model = LGBMClassifier(**params)

    accuracy = log_mlflow("LightGBM", model, X_train_resampled, X_test, y_train_resampled, y_test, params, trial.number)
    
    return accuracy

# Step 7: Run Optuna for XGBoost, log the best model only
def run_optuna_experiment():
    study = optuna.create_study(direction="maximize")
    study.optimize(objective_lightgbm, n_trials=100)

    # Get the best parameters and log only the best model
    best_params = study.best_params
    best_model = LGBMClassifier(**best_params,
                                random_state=42,
                                verbosity=-1)

    # Log the best model with MLflow, passing the algo_name as "xgboost"
    log_mlflow("LightGBM", best_model, X_train_resampled, X_test, y_train_resampled, y_test, best_params, "Best")


# Run the experiment for XGboost
run_optuna_experiment()

[I 2026-08-14 12:32:39,968] A new study created in memory with name: no-name-e7af9b40-a82d-43b1-9cf8-905b2f32dfd1
2026/08/14 12:33:46 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run Trial_0_LightGBM_SMOTE_TFIDF_Trigrams at: https://dagshub.com/rajeshxdatascience/yt-comment-sentiment-analysis.mlflow/#/experiments/6/runs/e0208920cb5a45139a9fbf13582b983b
🧪 View experiment at: https://dagshub.com/rajeshxdatascience/yt-comment-sentiment-analysis.mlflow/#/experiments/6


[I 2026-08-14 12:35:19,454] Trial 0 finished with value: 0.6633028774035183 and parameters: {'n_estimators': 263, 'learning_rate': 0.009479908370151566, 'max_depth': 5, 'num_leaves': 26, 'min_child_samples': 46, 'colsample_bytree': 0.9564614553482451, 'subsample': 0.7121466562940437, 'reg_alpha': 0.0018744393073063558, 'reg_lambda': 0.07344384622355302}. Best is trial 0 with value: 0.6633028774035183.
2026/08/14 12:37:28 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run Trial_1_LightGBM_SMOTE_TFIDF_Trigrams at: https://dagshub.com/rajeshxdatascience/yt-comment-sentiment-analysis.mlflow/#/experiments/6/runs/3dd352d0b5c04961a17acb036fa0fe48
🧪 View experiment at: https://dagshub.com/rajeshxdatascience/yt-comment-sentiment-analysis.mlflow/#/experiments/6


[I 2026-08-14 12:39:13,948] Trial 1 finished with value: 0.6268921314605209 and parameters: {'n_estimators': 213, 'learning_rate': 0.0048351137827323185, 'max_depth': 5, 'num_leaves': 89, 'min_child_samples': 44, 'colsample_bytree': 0.7382361112039417, 'subsample': 0.6425313252098941, 'reg_alpha': 0.0028453509419188285, 'reg_lambda': 0.01415213330546455}. Best is trial 0 with value: 0.6633028774035183.
2026/08/14 12:39:57 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


...

🏃 View run Trial_98_LightGBM_SMOTE_TFIDF_Trigrams at: https://dagshub.com/rajeshxdatascience/yt-comment-sentiment-analysis.mlflow/#/experiments/6/runs/0f7209d37e334c979f2ecd669557e092
🧪 View experiment at: https://dagshub.com/rajeshxdatascience/yt-comment-sentiment-analysis.mlflow/#/experiments/6


[I 2026-08-14 15:30:01,992] Trial 98 finished with value: 0.7891722351015955 and parameters: {'n_estimators': 283, 'learning_rate': 0.034372404859296296, 'max_depth': 10, 'num_leaves': 30, 'min_child_samples': 25, 'colsample_bytree': 0.5869606277814329, 'subsample': 0.5795596548400035, 'reg_alpha': 0.0031171226276577374, 'reg_lambda': 0.28633159751871406}. Best is trial 55 with value: 0.8400381835537979.
2026/08/14 15:31:16 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run Trial_99_LightGBM_SMOTE_TFIDF_Trigrams at: https://dagshub.com/rajeshxdatascience/yt-comment-sentiment-analysis.mlflow/#/experiments/6/runs/4bba9c70ed534416afe27b12b23c2ddd
🧪 View experiment at: https://dagshub.com/rajeshxdatascience/yt-comment-sentiment-analysis.mlflow/#/experiments/6


[I 2026-08-14 15:31:52,931] Trial 99 finished with value: 0.8082640120005454 and parameters: {'n_estimators': 271, 'learning_rate': 0.051424419605013066, 'max_depth': 10, 'num_leaves': 40, 'min_child_samples': 22, 'colsample_bytree': 0.551566470041674, 'subsample': 0.5570163589590302, 'reg_alpha': 0.0005158687374227132, 'reg_lambda': 0.19055163643422002}. Best is trial 55 with value: 0.8400381835537979.
2026/08/14 15:33:03 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run Trial_Best_LightGBM_SMOTE_TFIDF_Trigrams at: https://dagshub.com/rajeshxdatascience/yt-comment-sentiment-analysis.mlflow/#/experiments/6/runs/1d4d25d52b3c473ca5636aa79cf2224a
🧪 View experiment at: https://dagshub.com/rajeshxdatascience/yt-comment-sentiment-analysis.mlflow/#/experiments/6
